# m06_l01_unit

Exobean Engineering. Run the cells from top to bottom.

In [ ]:
# Finds the code companion. On Google Colab it downloads it once.
import glob, io, os, sys, urllib.request, zipfile
URL = "https://github.com/exobean/ai-agents-level1/releases/latest/download/shopassist-companion.zip"
if not os.path.isdir("shopassist"):
    if os.path.isdir(os.path.join("..", "shopassist")):
        os.chdir("..")  # opened from the notebooks folder on your computer
    else:
        found = glob.glob("*-companion")  # kept: your edited answers stay
        if not found and "google.colab" in sys.modules:
            try:
                data = io.BytesIO(urllib.request.urlopen(URL).read())
            except OSError:  # network blocks the download: upload the zip
                from google.colab import files
                data = next(iter(files.upload()))
            zipfile.ZipFile(data).extractall(".")
            found = glob.glob("*-companion")
        if found:
            os.chdir(found[0])
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print("Companion ready" if os.path.isdir("shopassist") else
      "Companion not found: see Lesson 0.1 of the book")


## Unit

In [ ]:
from shopassist.agent import StepLimitReached, run_agent
from shopassist.messages import Message, ToolCall
from shopassist.models import ScriptedModel
from shopassist.tools import GET_ORDER

ASK = Message("assistant", tool_calls=(
    ToolCall("c1", "get_order", {"order_id": "1042"}),))
DONE = Message("assistant", "It is packed.")


def test_tool_result_reaches_the_model():
    model = ScriptedModel([ASK, DONE])
    answer, msgs = run_agent(model, [GET_ORDER], "s", "#1042?")
    assert answer == "It is packed." and '"packed"' in msgs[2].text


def test_unknown_tool_is_reported():
    bad = Message("assistant", tool_calls=(ToolCall("c", "nope", {}),))
    model = ScriptedModel([bad, DONE])
    _, msgs = run_agent(model, [GET_ORDER], "s", "x")
    assert msgs[2].is_error


def test_runaway_is_stopped():
    try:
        run_agent(ScriptedModel([ASK]), [GET_ORDER], "s", "x",
                  max_steps=3)
    except StepLimitReached:
        return
    raise AssertionError("the loop did not stop")


for test in (test_tool_result_reaches_the_model,
             test_unknown_tool_is_reported, test_runaway_is_stopped):
    test()
    print("PASS", test.__name__)
